# Karachi AQI — Exploratory Data Analysis

Explores features from Hopsworks Feature Store: time series, correlations, seasonality, and outliers.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestRegressor
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

import config
from utils.feature_engineering import get_feature_columns
from utils.hopsworks_utils import read_feature_group

sns.set_theme(style="darkgrid")
%matplotlib inline

## 1. Data Loading from Hopsworks Feature Store

In [ ]:
df = read_feature_group()
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

## 2. Time Series Plots of AQI Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["timestamp"], df["aqi"], color="#60a5fa", linewidth=0.8)
ax.set_title("Karachi AQI Over Time")
ax.set_xlabel("Timestamp (UTC)")
ax.set_ylabel("AQI")
plt.tight_layout()
plt.show()

## 3. Correlation Heatmap

In [ ]:
numeric = df.select_dtypes(include=[np.number])
corr = numeric.corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. AQI Distribution by Hour, Day of Week, Month

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=df, x="hour", y="aqi", ax=axes[0])
axes[0].set_title("AQI by Hour of Day")
sns.boxplot(data=df, x="day_of_week", y="aqi", ax=axes[1])
axes[1].set_title("AQI by Day of Week")
sns.boxplot(data=df, x="month", y="aqi", ax=axes[2])
axes[2].set_title("AQI by Month (Monsoon Jul-Sep)")
plt.tight_layout()
plt.show()

## 5. Missing Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0")

## 6. Seasonal Decomposition

In [ ]:
ts = df.set_index("timestamp")["aqi"].asfreq("h").interpolate()
decomp = seasonal_decompose(ts, model="additive", period=24)
decomp.plot()
plt.suptitle("AQI Seasonal Decomposition (period=24h)", y=1.02)
plt.tight_layout()
plt.show()

## 7. ACF / PACF Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(ts.dropna(), lags=72, ax=axes[0])
axes[0].set_title("ACF — AQI")
plot_pacf(ts.dropna(), lags=48, ax=axes[1], method="ywm")
axes[1].set_title("PACF — AQI")
plt.tight_layout()
plt.show()

## 8. Pollutant Correlation Analysis

In [ ]:
pollutants = ["aqi", "pm25", "pm10", "no2", "o3", "co", "so2"]
poll_df = df[[c for c in pollutants if c in df.columns]]
sns.pairplot(poll_df.dropna(), diag_kind="kde", corner=True)
plt.suptitle("Pollutant Pairwise Relationships", y=1.02)
plt.show()

## 9. Quick Random Forest Feature Importance

In [ ]:
feature_cols = get_feature_columns(df)
train = df.dropna(subset=feature_cols + config.TARGET_COLUMNS)
X = train[feature_cols]
y = train["aqi_t24"]
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True).tail(15)
imp.plot(kind="barh", figsize=(8, 6), title="Top 15 RF Feature Importances (t24 target)")
plt.tight_layout()
plt.show()

## 10. Outliers and Anomalies

In [ ]:
q1, q3 = df["aqi"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["aqi"] < lower) | (df["aqi"] > upper)]
print(f"Outliers (IQR method): {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
outliers[["timestamp", "aqi", "pm25", "pm10"]].head(10)